In [12]:
import openai
import re
import time
import json

import numpy as np

from tqdm import tqdm
from pprint import pprint
from tenacity import retry, stop_after_attempt, wait_chain, wait_fixed

import os
from openai import AzureOpenAI

import math

import re
import math
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor
import traceback

In [13]:
endpoint = "https://pankajaiml.openai.azure.com/"
model_name = "gpt-35-turbo"
deployment = "gpt-35-turbo"
subscription_key = "REDACTED_AZURE_OPENAI_KEY"
api_version = "2024-12-01-preview"

client = AzureOpenAI(
    api_version=api_version,
    azure_endpoint=endpoint,
    api_key=subscription_key,
)

# Retry logic
@retry(wait=wait_chain(*[wait_fixed(3) for _ in range(3)] +
                       [wait_fixed(5) for _ in range(2)] +
                       [wait_fixed(10)]))
def completion_with_backoff(messages):
    return client.chat.completions.create(
        messages=messages,
        max_tokens=512,
        temperature=0.0,
        model=deployment
    )

In [14]:
def load_json(path):
    with open(path, 'r', encoding='utf-8') as reader:
        data = json.load(reader)  # Load the entire JSON file
    return data

dev_data = load_json('/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/testingDatasets/SVAMPsampled_train.json')
CoT_prompt_examples = open('/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/prompt_examples/CoT_prompt_examples.txt').read()
Standard_prompt_examples = open("/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/prompt_examples/standard_prompt_examples.txt").read()
CCoT_prompt_examples = open("/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/prompt_examples/CCoT_prompt_example.txt").read()

In [15]:
# === Metrics ===
acc = 0
total = 0

# === File Paths ===
output_path = '/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/logs/SVAMP/CoT.txt'
bad_output_path = output_path.replace('.txt', '_bad.txt')

# === Cleaning & Truncation Utility ===
def clean_and_truncate(value_str):
    cleaned = re.sub(r'[^\d\.\-]', '', value_str)
    try:
        num = float(cleaned)
        return num
    except ValueError:
        return None

# === Function to Process a Single Entry ===
def process_entry(d):
    try:
        q = d['question']
        a = float(d['correct'])  # Ground truth

        prompt_q = (
            CoT_prompt_examples +
            '\nQ: ' + q + " Think step by step. Write your answer as: the answer is <answer>"
        )

        messages = [
            {"role": "system", "content": "Your goal is to answer these math questions accurately."},
            {"role": "user", "content": prompt_q}
        ]

        response = completion_with_backoff(messages)
        ans_model = response.choices[0].message.content.strip()

        # === Extract Answer
        match = re.search(r'the answer is\s*([-\d\.,\$\%]+)', ans_model, re.IGNORECASE)
        if match:
            extracted_raw = match.group(1).strip().rstrip('.')
            extracted = clean_and_truncate(extracted_raw)
        else:
            extracted = None

        # === Log Block
        log_block = (
            f'Q: {q}\n'
            f'A_model:\n{ans_model}\n'
            f'Extracted:\n{extracted}\n'
            f'A:\n{a}\n\n'
        )

        # === Determine Correctness
        if extracted is not None and math.isclose(extracted, a, rel_tol=1e-4):
            return "correct", log_block
        else:
            return "incorrect", "❌ Incorrect or Invalid\n" + log_block

    except Exception as e:
        return "error", f"Error processing entry: {d}\nException: {str(e)}\n\n"

# === Main Parallel Processing ===
with open(output_path, 'w') as fd, open(bad_output_path, 'w') as bad_fd:
    with ThreadPoolExecutor() as executor:
        futures = [executor.submit(process_entry, d) for d in dev_data]
        for future in tqdm(futures):
            result_type, log = future.result()
            if result_type == "correct":
                acc += 1
                fd.write(log)
            elif result_type == "incorrect":
                bad_fd.write(log)
            elif result_type == "error":
                bad_fd.write(log)
            total += 1
            print(f"Accuracy: {acc} / {total} = {acc / total:.2%}")

  0%|          | 1/205 [00:00<01:35,  2.13it/s]

Accuracy: 1 / 1 = 100.00%


  1%|          | 2/205 [00:01<02:16,  1.48it/s]

Accuracy: 2 / 2 = 100.00%


  1%|▏         | 3/205 [00:01<01:34,  2.14it/s]

...

Accuracy: 148 / 187 = 79.14%
Accuracy: 149 / 188 = 79.26%
Accuracy: 150 / 189 = 79.37%
Accuracy: 151 / 190 = 79.47%
Accuracy: 152 / 191 = 79.58%


 94%|█████████▎| 192/205 [02:05<00:02,  5.48it/s]

Accuracy: 153 / 192 = 79.69%


100%|██████████| 205/205 [02:06<00:00,  1.62it/s]

Accuracy: 154 / 193 = 79.79%
Accuracy: 155 / 194 = 79.90%
Accuracy: 156 / 195 = 80.00%
Accuracy: 157 / 196 = 80.10%
Accuracy: 158 / 197 = 80.20%
Accuracy: 158 / 198 = 79.80%
Accuracy: 159 / 199 = 79.90%
Accuracy: 160 / 200 = 80.00%
Accuracy: 160 / 201 = 79.60%
Accuracy: 161 / 202 = 79.70%
Accuracy: 162 / 203 = 79.80%
Accuracy: 163 / 204 = 79.90%
Accuracy: 164 / 205 = 80.00%


In [16]:
# === Metrics ===
acc = 0
total = 0

# === File Output Paths ===
output_path = '/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/logs/SVAMP/standard.txt'
bad_output_path = output_path.replace('.txt', '_bad.txt')

# === Cleaning & Truncation Utility ===
def clean_and_truncate(value_str):
    """Remove $, %, commas, etc. and round to 4 decimal places"""
    cleaned = re.sub(r'[^\d\.\-]', '', value_str)  # Keep digits, decimal, minus
    try:
        num = float(cleaned)
        return round(num, 4)
    except ValueError:
        return None

# === Function to Process a Single Entry ===
def process_entry(d):
    try:
        q = d['question']
        a = float(d['correct'])  # Ground truth

        prompt_q = (
            Standard_prompt_examples +
            '\nAnswer this question: ' + q + " Write your answer as: the answer is <answer>"
        )

        messages = [
            {"role": "system", "content": "Your goal is to answer these math questions correctly."},
            {"role": "user", "content": prompt_q}
        ]

        response = completion_with_backoff(messages)
        ans_model = response.choices[0].message.content.strip()

        # === Improved Answer Extraction ===
        match = re.search(r'the answer is\s*([-\d\.,\$\%]+)', ans_model, re.IGNORECASE)
        if match:
            extracted_raw = match.group(1).strip().rstrip('.')
            extracted = clean_and_truncate(extracted_raw)
        else:
            extracted = None

        # === Log Block
        log_block = (
            f'Q: {q}\n'
            f'A_model:\n{ans_model}\n'
            f'Extracted:\n{extracted}\n'
            f'A:\n{a}\n\n'
        )

        if extracted is not None and math.isclose(extracted, a, rel_tol=1e-4):
            return "correct", log_block
        else:
            return "incorrect", "❌ Incorrect or Invalid\n" + log_block

    except Exception as e:
        return "error", f"Error processing entry: {d}\nException: {str(e)}\n\n"

# === Main Parallel Processing ===
results = []
with open(output_path, 'w') as fd, open(bad_output_path, 'w') as bad_fd:
    with ThreadPoolExecutor() as executor:
        futures = [executor.submit(process_entry, d) for d in dev_data]
        for future in tqdm(futures):
            result_type, log = future.result()
            if result_type == "correct":
                global acc
                acc += 1
                fd.write(log)
            elif result_type == "incorrect":
                bad_fd.write(log)
            total += 1
            print(f"Accuracy: {acc} / {total} = {acc / total:.2%}")
    fd.write(f"\nFinal Accuracy: {acc} / {total} = {acc / total:.2%}\n")
    fd.write(f"Final Accuracy: {acc} / {total} = {acc / total:.2%}\n") 

  0%|          | 1/205 [00:00<00:48,  4.22it/s]

Accuracy: 1 / 1 = 100.00%
Accuracy: 1 / 2 = 50.00%
Accuracy: 2 / 3 = 66.67%
Accuracy: 2 / 4 = 50.00%
Accuracy: 3 / 5 = 60.00%
Accuracy: 4 / 6 = 66.67%
Accuracy: 5 / 7 = 71.43%
Accuracy: 5 / 8 = 62.50%
Accuracy: 6 / 9 = 66.67%
Accuracy: 6 / 10 = 60.00%
Accuracy: 7 / 11 = 63.64%
Accuracy: 8 / 12 = 66.67%


  6%|▋         | 13/205 [00:00<00:06, 29.36it/s]

Accuracy: 9 / 13 = 69.23%
Accuracy: 10 / 14 = 71.43%
Accuracy: 11 / 15 = 73.33%
Accuracy: 12 / 16 = 75.00%
Accuracy: 13 / 17 = 76.47%
Accuracy: 14 / 18 = 77.78%
Accuracy: 15 / 19 = 78.95%
Accuracy: 16 / 20 = 80.00%
Accuracy: 17 / 21 = 80.95%
Accuracy: 18 / 22 = 81.82%
Accuracy: 19 / 23 = 82.61%
Accuracy: 20 / 24 = 83.33%


 12%|█▏        | 25/205 [00:55<08:00,  2.67s/it]

...

Accuracy: 100 / 137 = 72.99%
Accuracy: 101 / 138 = 73.19%
Accuracy: 102 / 139 = 73.38%
Accuracy: 103 / 140 = 73.57%
Accuracy: 104 / 141 = 73.76%
Accuracy: 105 / 142 = 73.94%
Accuracy: 106 / 143 = 74.13%
Accuracy: 107 / 144 = 74.31%
Accuracy: 108 / 145 = 74.48%
Accuracy: 108 / 146 = 73.97%
Accuracy: 109 / 147 = 74.15%
Accuracy: 110 / 148 = 74.32%
Accuracy: 110 / 149 = 73.83%
Accuracy: 111 / 150 = 74.00%
Accuracy: 112 / 151 = 74.17%
Accuracy: 113 / 152 = 74.34%
Accuracy: 114 / 153 = 74.51%
Accuracy: 115 / 154 = 74.68%
Accuracy: 116 / 155 = 74.84%
Accuracy: 117 / 156 = 75.00%
Accuracy: 118 / 157 = 75.16%
Accuracy: 118 / 158 = 74.68%
Accuracy: 119 / 159 = 74.84%
Accuracy: 119 / 160 = 74.38%
Accuracy: 120 / 161 = 74.53%
Accuracy: 120 / 162 = 74.07%
Accuracy: 121 / 163 = 74.23%
Accuracy: 122 / 164 = 74.39%
Accuracy: 122 / 165 = 73.94%
Accuracy: 123 / 166 = 74.10%
Accuracy: 124 / 167 = 74.25%


 83%|████████▎ | 170/205 [02:07<00:18,  1.94it/s]

Accuracy: 125 / 168 = 74.40%
Accuracy: 126 / 169 = 74.56%
Accuracy: 127 / 170 = 74.71%
Accuracy: 128 / 171 = 74.85%
Accuracy: 129 / 172 = 75.00%
Accuracy: 129 / 173 = 74.57%
Accuracy: 130 / 174 = 74.71%
Accuracy: 131 / 175 = 74.86%
Accuracy: 132 / 176 = 75.00%
Accuracy: 133 / 177 = 75.14%
Accuracy: 134 / 178 = 75.28%
Accuracy: 135 / 179 = 75.42%
Accuracy: 136 / 180 = 75.56%
Accuracy: 137 / 181 = 75.69%
Accuracy: 138 / 182 = 75.82%
Accuracy: 139 / 183 = 75.96%
Accuracy: 140 / 184 = 76.09%
Accuracy: 141 / 185 = 76.22%


100%|██████████| 205/205 [02:09<00:00,  1.58it/s]

Accuracy: 142 / 186 = 76.34%
Accuracy: 143 / 187 = 76.47%
Accuracy: 144 / 188 = 76.60%
Accuracy: 145 / 189 = 76.72%
Accuracy: 146 / 190 = 76.84%
Accuracy: 147 / 191 = 76.96%
Accuracy: 148 / 192 = 77.08%
Accuracy: 148 / 193 = 76.68%
Accuracy: 149 / 194 = 76.80%
Accuracy: 150 / 195 = 76.92%
Accuracy: 151 / 196 = 77.04%
Accuracy: 152 / 197 = 77.16%
Accuracy: 153 / 198 = 77.27%
Accuracy: 154 / 199 = 77.39%
Accuracy: 155 / 200 = 77.50%
Accuracy: 156 / 201 = 77.61%
Accuracy: 157 / 202 = 77.72%
Accuracy: 158 / 203 = 77.83%
Accuracy: 159 / 204 = 77.94%
Accuracy: 160 / 205 = 78.05%


In [18]:
# === Metrics ===
acc = 0
total = 0

# === File Output Paths ===
output_path = '/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/logs/SVAMP/complexCoT.txt'
bad_output_path = output_path.replace('.txt', '_bad.txt')
error_log_path = output_path.replace('.txt', '_errors.txt')

# === Cleaning & Truncation Utility ===
def clean_and_truncate(value_str):
    """Remove $, %, commas, etc. and round to 4 decimal places"""
    cleaned = re.sub(r'[^\d\.\-]', '', value_str)
    try:
        num = float(cleaned)
        return round(num, 4)
    except ValueError:
        return None

# === Function to Process a Single Entry ===
def process_entry(idx, d):
    global acc, total
    try:
        q = d['question']
        a = float(d['correct'])  # Ground truth answer

        # === Prompt Setup for Complex CoT ===
        prompt_q = (
            CCoT_prompt_examples +
            "\nQ: " + q + "\n\n"
            "Please reason through this problem using a complex, multi-step chain of thought:\n"
            "Step 1: Clearly state all given information and any assumptions.\n"
            "Step 2: Propose two different methods to solve the problem, briefly outlining the logic of each.\n"
            "Step 3: For each method, work through all intermediate steps in detail, showing calculations, checks, and potential pitfalls.\n"
            "Step 4: Evaluate and compare the two methods—discussing which is better based on clarity, reliability, or efficiency.\n"
            "Step 5: Choose the better method and use it to solve the problem, showing all steps.\n"
            "Step 6: Double-check the solution for errors or unreasonable results.\n"
            "Finish your response with: the answer is <answer>"
        )

        messages = [
            {
                "role": "system",
                "content": (
                    "Your goal is to answer the question using a complex, coherent, step by step thoughts, answering the questions correctly.\n"
                )
            },
            {"role": "user", "content": prompt_q}
        ]

        # === Get Response ===
        response = completion_with_backoff(messages)
        ans_model = response.choices[0].message.content.strip()

        # === Improved Answer Extraction ===
        match = re.search(r'the answer is\s*([-\d\.,\$\%]+)', ans_model, re.IGNORECASE)
        if match:
            extracted_raw = match.group(1).strip().rstrip('.')
            extracted = clean_and_truncate(extracted_raw)
        else:
            extracted = None

        # === Log Block
        log_block = (
            f'Q: {q}\n'
            f'A_model:\n{ans_model}\n'
            f'Extracted:\n{extracted}\n'
            f'A:\n{a}\n\n'
        )

        if extracted is not None and math.isclose(extracted, a, rel_tol=1e-4):
            return "correct", log_block
        else:
            return "incorrect", "❌ Incorrect or Invalid\n" + log_block

    except Exception as e:
        # Log the error and the problematic entry
        error_log = f"Error at index {idx}:\nData: {d}\nTraceback:\n{traceback.format_exc()}\n\n"
        return "error", error_log

# === Main Parallel Processing ===
results = []
start_index = 0  # Start processing from question 127
with open(output_path, 'a') as fd, open(bad_output_path, 'a') as bad_fd, open(error_log_path, 'a') as error_fd:
    with ThreadPoolExecutor() as executor:
        futures = [executor.submit(process_entry, idx, d) for idx, d in enumerate(dev_data[start_index:], start=start_index)]
        for future in tqdm(futures):
            result_type, log = future.result()
            if result_type == "correct":
                acc += 1
                fd.write(log)
            elif result_type == "incorrect":
                bad_fd.write(log)
            elif result_type == "error":
                error_fd.write(log)
            total += 1
            print(f"Accuracy: {acc} / {total} = {acc / total:.2%}")

    # Final accuracy report
    fd.write(f"\nFinal Accuracy: {acc} / {total} = {acc / total:.2%}\n")
    fd.write(f"Final Accuracy: {acc} / {total} = {acc / total:.2%}\n")  # Write to the output file


  0%|          | 1/205 [00:04<15:40,  4.61s/it]

Accuracy: 1 / 1 = 100.00%
Accuracy: 2 / 2 = 100.00%
Accuracy: 3 / 3 = 100.00%


  2%|▏         | 4/205 [00:08<06:45,  2.02s/it]

Accuracy: 3 / 4 = 75.00%
Accuracy: 4 / 5 = 80.00%
Accuracy: 5 / 6 = 83.33%
Accuracy: 6 / 7 = 85.71%
Accuracy: 7 / 8 = 87.50%
Accuracy: 8 / 9 = 88.89%
Accuracy: 9 / 10 = 90.00%
Accuracy: 10 / 11 = 90.91%
Accuracy: 11 / 12 = 91.67%
Accuracy: 11 / 13 = 84.62%
Accuracy: 12 / 14 = 85.71%
Accuracy: 12 / 15 = 80.00%
Accuracy: 13 / 16 = 81.25%
Accuracy: 14 / 17 = 82.35%
Accuracy: 15 / 18 = 83.33%
Accuracy: 16 / 19 = 84.21%
Accuracy: 17 / 20 = 85.00%
Accuracy: 18 / 21 = 85.71%
Accuracy: 19 / 22 = 86.36%
Accuracy: 20 / 23 = 86.96%
Accuracy: 21 / 24 = 87.50%
Accuracy: 22 / 25 = 88.00%


 14%|█▍        | 29/205 [00:09<00:37,  4.71it/s]

...

Accuracy: 144 / 194 = 74.23%
Accuracy: 145 / 195 = 74.36%
Accuracy: 146 / 196 = 74.49%


 96%|█████████▌| 197/205 [03:17<00:02,  3.25it/s]

Accuracy: 147 / 197 = 74.62%


100%|██████████| 205/205 [04:07<00:00,  1.20s/it]

Accuracy: 148 / 198 = 74.75%
Accuracy: 149 / 199 = 74.87%
Accuracy: 150 / 200 = 75.00%
Accuracy: 150 / 201 = 74.63%
Accuracy: 151 / 202 = 74.75%
Accuracy: 152 / 203 = 74.88%
Accuracy: 152 / 204 = 74.51%
Accuracy: 153 / 205 = 74.63%
